In [1]:
import os, re, cv2, torch, pandas as pd
import torchvision.models as models
import torchvision.transforms as transforms
from pathlib import Path


image_folder = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_test")


def get_id(filename: str) -> int:
    return int(re.findall(r'\d+', Path(filename).stem)[0])


image_files = sorted(
    [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.png'))],
    key=get_id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_config = [
    ('resnet18',         models.resnet18,          'features_Resnet18_TEST.xlsx'),
    ('resnet50',         models.resnet50,          'features_Resnet50_TEST.xlsx'),
    ('efficientnet_b0',  models.efficientnet_b0,   'features_EfficientNetB0_TEST.xlsx'),
    ('mobilenet_v3_small', models.mobilenet_v3_small, 'features_MobileNetV3Small_TEST.xlsx'),
    ('vgg16',            models.vgg16,             'features_VGG16_TEST.xlsx'),
    ('densenet121',      models.densenet121,       'features_DenseNet121_TEST.xlsx'),
    ('inception_v3',     models.inception_v3,      'features_InceptionV3_TEST.xlsx'),
    ('vit_tiny', lambda: torch.hub.load('facebookresearch/dino:main', 'dino_vits16'), 'features_ViTTiny_TEST.xlsx')
]

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

for model_name, model_fn, output_name in models_config:

    if 'vit' in model_name:
        model = model_fn().to(device).eval()
    else:
        model = model_fn(pretrained=True).to(device).eval()
    
    all_rows = []          
    
    for img_file in image_files:
        img_path = image_folder / img_file
        img_id   = get_id(img_file)     
        
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            feats = model(img_tensor).cpu().numpy().flatten()
        

        row = [img_id] + feats.tolist()
        all_rows.append(row)
    

    df = pd.DataFrame(all_rows)
    df.to_excel(output_name, index=False, header=False)
    print(f"{output_name} saved ✅")


In [8]:
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder
# from lazypredict.Supervised import LazyClassifier
# 
# # --- بارگذاری داده‌های آموزشی ---
# train_feat_path = "features_Resnet50_TEST.xlsx"  # ویژگی‌های آموزشی
# train_feat_df = pd.read_excel(train_feat_path, header=None)
# train_feat_df.rename(columns={0: "name"}, inplace=True)
# train_feat_df["name"] = train_feat_df["name"].astype(str) + ".jpg"
# 
# # --- بارگذاری داده‌های تست ---
# test_feat_path = "features_Resnet50_TEST.xlsx"  # ویژگی‌های تست
# test_feat_df = pd.read_excel(test_feat_path, header=None)
# test_feat_df.rename(columns={0: "name"}, inplace=True)
# test_feat_df["name"] = test_feat_df["name"].astype(str) + ".jpg"
# 
# # --- بارگذاری لیبل‌ها ---
# label_df = pd.read_csv(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv")
# label_df["name"] = label_df["name"].astype(str)
# 
# # --- ادغام داده‌ها ---
# train_data = train_feat_df.merge(label_df, on="name", how="inner")
# test_data = test_feat_df.merge(label_df, on="name", how="inner")
# 
# # --- آماده‌سازی داده‌ها ---
# X_train = train_data.drop(columns=["name", "category", "type", "grade"])
# y_train = LabelEncoder().fit_transform(train_data["grade"])
# 
# X_test = test_data.drop(columns=["name", "category", "type", "grade"])
# y_test = LabelEncoder().fit_transform(test_data["grade"])
# 
# # --- آموزش و ارزیابی ---
# clf = LazyClassifier(verbose=1, ignore_warnings=True, random_state=42)
# models, _ = clf.fit(X_train, X_test, y_train, y_test)
# print(models.sort_values("Balanced Accuracy", ascending=False).head(10))


import pandas as pd
from sklearn.preprocessing import LabelEncoder
from lazypredict.Supervised import LazyClassifier

# لیست مدل‌ها و مسیر فایل‌های ویژگی
MODELS_CONFIG = [
    ('ResNet18', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet18.xlsx', 'features_Resnet18_TEST.xlsx'),
    ('ResNet50', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet50.xlsx', 'features_Resnet50_TEST.xlsx'),
    ('EfficientNetB0', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_EfficientNetB0.xlsx', 'features_EfficientNetB0_TEST.xlsx'),
    ('MobileNetV3', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_MobileNetV3Small.xlsx', 'features_MobileNetV3Small_TEST.xlsx'),
    ('VGG16', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_VGG16.xlsx', 'features_VGG16_TEST.xlsx'),
    ('DenseNet121', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_DenseNet121.xlsx', 'features_DenseNet121_TEST.xlsx'),
    ('InceptionV3', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_InceptionV3.xlsx', 'features_InceptionV3_TEST.xlsx'),
    ('ViTTiny', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_ViTTiny.xlsx', 'features_ViTTiny_TEST.xlsx')
]
# r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet50.xlsx"
# تنظیمات مشترک
LABEL_PATH = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv"
RESULTS = []

def process_model(model_name: str, train_feat_path: str, test_feat_path: str):
    # بارگذاری ویژگی‌ها
    train_feat = pd.read_excel(train_feat_path, header=None).rename(columns={0: 'name'})
    test_feat = pd.read_excel(test_feat_path, header=None).rename(columns={0: 'name'})
    
    # افزودن پسوند .jpg
    train_feat['name'] = train_feat['name'].astype(str) + '.jpg'
    test_feat['name'] = test_feat['name'].astype(str) + '.jpg'
    
    # بارگذاری و ادغام لیبل‌ها
    labels = pd.read_csv(LABEL_PATH).astype({'name': 'str'})
    train_data = train_feat.merge(labels, on='name', how='inner')
    test_data = test_feat.merge(labels, on='name', how='inner')
    
    # آماده‌سازی داده‌ها
    X_train = train_data.drop(columns=['name', 'category', 'type', 'grade'])
    y_train = LabelEncoder().fit_transform(train_data['category'])
    X_test = test_data.drop(columns=['name', 'category', 'type', 'grade'])
    y_test = LabelEncoder().fit_transform(test_data['category'])
    
    # آموزش و ارزیابی
    clf = LazyClassifier(verbose=0, ignore_warnings=True, random_state=42)
    models, _ = clf.fit(X_train, X_test, y_train, y_test)
    return models

# پردازش تمام مدل‌ها
for model_name, train_path, test_path in MODELS_CONFIG:
    try:
        model_results = process_model(model_name, train_path, test_path)
        best_result = model_results.iloc[0].to_dict()
        RESULTS.append({
            'Model': model_name,
            'Best Algorithm': model_results.index[0],
            'Accuracy': best_result['Accuracy'],
            'Balanced Accuracy': best_result['Balanced Accuracy']
        })
        print(f"{model_name} evaluation completed ✅")
    except Exception as e:
        print(f"Error in {model_name}: {str(e)}")

# نمایش نتایج نهایی
results_df = pd.DataFrame(RESULTS)
print("\n" + "="*60)
print("Final Ranking (Sorted by Balanced Accuracy):")
print("="*60)
print(results_df.sort_values('Balanced Accuracy', ascending=False).reset_index(drop=True))

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from lazypredict.Supervised import LazyClassifier

# لیست مدل‌ها و مسیر فایل‌های ویژگی
MODELS_CONFIG = [
    ('ResNet18', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet18.xlsx', 'features_Resnet18_TEST.xlsx'),
    ('ResNet50', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet50.xlsx', 'features_Resnet50_TEST.xlsx'),
    ('EfficientNetB0', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_EfficientNetB0.xlsx', 'features_EfficientNetB0_TEST.xlsx'),
    ('MobileNetV3', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_MobileNetV3Small.xlsx', 'features_MobileNetV3Small_TEST.xlsx'),
    ('VGG16', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_VGG16.xlsx', 'features_VGG16_TEST.xlsx'),
    ('DenseNet121', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_DenseNet121.xlsx', 'features_DenseNet121_TEST.xlsx'),
    ('InceptionV3', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_InceptionV3.xlsx', 'features_InceptionV3_TEST.xlsx'),
    ('ViTTiny', r'C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_ViTTiny.xlsx', 'features_ViTTiny_TEST.xlsx')
]
# r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\Extracted_features\features_Resnet50.xlsx"
# تنظیمات مشترک
LABEL_PATH = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv"
RESULTS = []

def process_model(model_name: str, train_feat_path: str, test_feat_path: str):
    # بارگذاری ویژگی‌ها
    train_feat = pd.read_excel(train_feat_path, header=None).rename(columns={0: 'name'})
    test_feat = pd.read_excel(test_feat_path, header=None).rename(columns={0: 'name'})
    
    # افزودن پسوند .jpg
    train_feat['name'] = train_feat['name'].astype(str) + '.jpg'
    test_feat['name'] = test_feat['name'].astype(str) + '.jpg'
    
    # بارگذاری و ادغام لیبل‌ها
    labels = pd.read_csv(LABEL_PATH).astype({'name': 'str'})
    train_data = train_feat.merge(labels, on='name', how='inner')
    test_data = test_feat.merge(labels, on='name', how='inner')
    
    # آماده‌سازی داده‌ها
    X_train = train_data.drop(columns=['name', 'category', 'type', 'grade'])
    y_train = LabelEncoder().fit_transform(train_data['category'])
    X_test = test_data.drop(columns=['name', 'category', 'type', 'grade'])
    y_test = LabelEncoder().fit_transform(test_data['grade'])
    
    # آموزش و ارزیابی
    clf = LazyClassifier(verbose=0, ignore_warnings=True, random_state=42)
    models, _ = clf.fit(X_train, X_test, y_train, y_test)
    return models

# پردازش تمام مدل‌ها
for model_name, train_path, test_path in MODELS_CONFIG:
    try:
        model_results = process_model(model_name, train_path, test_path)
        best_result = model_results.iloc[0].to_dict()
        RESULTS.append({
            'Model': model_name,
            'Best Algorithm': model_results.index[0],
            'Accuracy': best_result['Accuracy'],
            'Balanced Accuracy': best_result['Balanced Accuracy']
        })
        print(f"{model_name} evaluation completed ✅")
    except Exception as e:
        print(f"Error in {model_name}: {str(e)}")

# نمایش نتایج نهایی
results_df = pd.DataFrame(RESULTS)
print("\n" + "="*60)
print("Final Ranking (Sorted by Balanced Accuracy):")
print("="*60)
print(results_df.sort_values('Balanced Accuracy', ascending=False).reset_index(drop=True))